# Capstone: A Complete Analysis

**Computational Text Analysis Pack · Module 12 — Capstone**

*Technique:* The whole workflow, on a corpus and question of your own

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/17-capstone/12_capstone.ipynb)

Data: `masakhanews.csv`, `colonial_texts.csv`, `african_legal.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv', 'colonial_texts.csv', 'african_legal.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
df_colonial = pd.read_csv('colonial_texts.csv')
df_legal = pd.read_csv('african_legal.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Everything, once, end to end — and then you replace the parameters with your own.

The structure below is the structure of a paper. Work down it in order. The only
cells you should need to edit are the ones marked with `←`.

**The deliverable is not the code.** It is: one question, one figure, one
paragraph of finding, one paragraph of limits.

## The tooling

Everything the course has used: **pandas**, **`re`**, **`Counter`**,
**scikit-learn**, **matplotlib**. The capstone deliberately adds nothing new.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
from collections import Counter

def frequencies(texts):
    """Token counts across a set of documents."""
    counts = Counter()
    for t in texts:
        counts.update(tokenise(t))
    return counts

def freq_frame(counts, top=25):
    total = sum(counts.values())
    rows = [{"rank": i, "word": w, "count": n, "per_1000": n / total * 1000}
            for i, (w, n) in enumerate(counts.most_common(top), start=1)]
    return pd.DataFrame(rows)

In [ ]:
def log_likelihood(counts_a, counts_b, min_count=10):
    """Words that distinguish corpus A from corpus B.

    The G-squared statistic corpus linguists call *keyness*. A word scores high
    when it is far more common in one corpus than its overall rate predicts.
    Positive `diff` means over-used in A.
    """
    na, nb = sum(counts_a.values()), sum(counts_b.values())
    rows = []
    for w in set(counts_a) | set(counts_b):
        o1, o2 = counts_a.get(w, 0), counts_b.get(w, 0)
        if o1 + o2 < min_count:
            continue
        e1 = na * (o1 + o2) / (na + nb)
        e2 = nb * (o1 + o2) / (na + nb)
        g2 = 2 * ((o1 * np.log(o1 / e1) if o1 else 0)
                  + (o2 * np.log(o2 / e2) if o2 else 0))
        rows.append({"word": w, "count_a": o1, "count_b": o2,
                     "per_1000_a": o1 / na * 1000, "per_1000_b": o2 / nb * 1000,
                     "keyness": g2, "over_used_in": "A" if o1 / na > o2 / nb else "B"})
    out = pd.DataFrame(rows).sort_values("keyness", ascending=False)
    return out.reset_index(drop=True)

In [ ]:
def kwic(texts, term, width=45, limit=12):
    """Keyword in context: every occurrence with the words either side."""
    pattern = re.compile(rf"\b{re.escape(term)}\b", re.IGNORECASE)
    rows = []
    for doc_id, text in enumerate(texts):
        s = str(text)
        for m in pattern.finditer(s):
            left = s[max(0, m.start() - width):m.start()].replace("\n", " ")
            right = s[m.end():m.end() + width].replace("\n", " ")
            rows.append({"doc": doc_id, "left": left.rjust(width),
                         "match": m.group(0), "right": right})
            if len(rows) >= limit:
                return pd.DataFrame(rows)
    return pd.DataFrame(rows)

## 1. The question

Three parts, from 01a: a corpus, a countable feature, a comparison.

In [ ]:
# ← EDIT THIS BLOCK. Everything below follows from it.
QUESTION = ("Does health reporting in Nigerian-language news use a different "
            "vocabulary from political reporting in the same languages?")

CORPUS = "news"             # "news" | "colonial" | "legal"
GROUP_COLUMN = "category"   # the column that splits target from reference
TARGET_VALUE = "health"     # ← target
REFERENCE_VALUE = "politics"  # ← reference; None means "everything else"
RESTRICT = ("is_nigerian", True)   # optional (column, value) filter; None for all

print(QUESTION)
print()
print(f"corpus     : {CORPUS}")
print(f"target     : {GROUP_COLUMN} == {TARGET_VALUE!r}")
print(f"reference  : {GROUP_COLUMN} == {REFERENCE_VALUE!r}"
      if REFERENCE_VALUE else f"reference  : everything else")
print(f"restricted : {RESTRICT}")

## 2. Build and document the corpus

In [ ]:
source = {"news": df, "colonial": df_colonial, "legal": df_legal}[CORPUS]
work = source.copy()
if RESTRICT:
    work = work[work[RESTRICT[0]] == RESTRICT[1]]

target = work[work[GROUP_COLUMN] == TARGET_VALUE]
reference = (work[work[GROUP_COLUMN] == REFERENCE_VALUE] if REFERENCE_VALUE
             else work[work[GROUP_COLUMN] != TARGET_VALUE])

print(f"target    : {len(target):>6,} documents")
print(f"reference : {len(reference):>6,} documents")
print()
if min(len(target), len(reference)) < 30:
    print("⚠ fewer than 30 documents on one side. Any keyness result will be")
    print("  unstable. Widen the definition or choose a different comparison.")
else:
    print("Both sides have enough documents for the comparison to be meaningful.")

In [ ]:
# Document the corpus properly — this table goes into your paper.
def describe(name, frame):
    toks = 0
    for t in frame["text"].head(400):
        toks += len(tokenise(t))
    scale = len(frame) / max(min(len(frame), 400), 1)
    return {"corpus": name, "documents": len(frame),
            "median words": frame["n_words"].median(),
            "estimated tokens": int(toks * scale)}

print(pd.DataFrame([describe("target", target), describe("reference", reference)])
      .to_string(index=False, float_format=lambda v: f"{v:,.0f}"))

## 3. The absence audit

Before any result. What is not here, and is the gap systematic?

In [ ]:
print("What is in the corpus:")
for col in ("language", GROUP_COLUMN):
    if col in work.columns:
        print(f"\n{col}:")
        print(work[col].value_counts().head(10).to_string())
print()
print("Write three sentences here, in a markdown cell of your own:")
print("  1. How was this corpus assembled, and by what filter?")
print("  2. Who or what is systematically absent from it?")
print("  3. Does that absence bear on your question? How?")

## 4. Prepare — and say what you did

In [ ]:
import unicodedata

MIN_COUNT = 25              # ← from 02b: below this, the statistic is noise

def prepare(text):
    return re.findall(r"\w+", unicodedata.normalize("NFC", str(text)).lower())

counts_t = Counter()
counts_r = Counter()
for t in target["text"]:
    counts_t.update(prepare(t))
for t in reference["text"]:
    counts_r.update(prepare(t))

print(f"target   : {sum(counts_t.values()):>10,} tokens  {len(counts_t):>8,} types")
print(f"reference: {sum(counts_r.values()):>10,} tokens  {len(counts_r):>8,} types")
print()
print("Methods sentence you can paste:")
print(f"  Texts were normalised to Unicode NFC and tokenised on \\w+ "
      "(Unicode-aware,")
print(f"  preserving diacritics); no stopword list was applied, and types "
      f"occurring")
print(f"  fewer than {MIN_COUNT} times were excluded from the keyness test.")

## 5. Compare

In [ ]:
keys = log_likelihood(counts_t, counts_r, min_count=MIN_COUNT)
keys["log_ratio"] = np.log2(keys["per_1000_a"].clip(lower=1e-4)
                            / keys["per_1000_b"].clip(lower=1e-4))

print(f"Distinctive of {TARGET_VALUE}:")
print(keys[keys["over_used_in"] == "A"].head(15)[
      ["word", "count_a", "count_b", "per_1000_a", "per_1000_b",
       "keyness", "log_ratio"]]
      .to_string(index=False, float_format=lambda v: f"{v:.2f}"))

## 6. One figure

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 6.5))
ax.scatter(keys["log_ratio"], keys["keyness"], s=15, alpha=0.28, color="grey")
notable = keys[(keys["keyness"] > keys["keyness"].quantile(0.97))
               & (keys["log_ratio"].abs() > 0.8)]
ax.scatter(notable["log_ratio"], notable["keyness"], s=28, color="indianred")
for _, r in notable.head(20).iterrows():
    ax.annotate(r["word"], (r["log_ratio"], r["keyness"]), fontsize=8,
                xytext=(4, 2), textcoords="offset points")
ax.axvline(0, color="grey", lw=0.8)
ref_label = REFERENCE_VALUE if REFERENCE_VALUE else "reference"
ax.set_xlabel(f"← {ref_label}        log₂ rate ratio        {TARGET_VALUE} →")
ax.set_ylabel("keyness (log-likelihood G²)")
ax.set_title(QUESTION[:88], fontsize=10)
plt.tight_layout(); plt.show()

## 7. Read the lines

Rung 5 of the evidence ladder. Never publish a keyness word you have not seen in
context.

In [ ]:
CHECK = notable.iloc[0]["word"] if len(notable) else keys.iloc[0]["word"]
print(f"concordance for '{CHECK}' in the target corpus:\n")
for _, r in kwic(target["text"], CHECK, width=42, limit=10).iterrows():
    print(f"{r['left']} [{r['match']}] {r['right']}")

## 8. Check one alternative explanation

In [ ]:
# Does the reference choice drive the result? Swap in a different reference
# and see which words survive. (04c: a finding that survives is much stronger.)
alt_reference = work[work[GROUP_COLUMN] != TARGET_VALUE]
counts_alt = Counter()
for t in alt_reference["text"]:
    counts_alt.update(prepare(t))
keys_alt = log_likelihood(counts_t, counts_alt, min_count=MIN_COUNT)

a = set(keys[keys["over_used_in"] == "A"].head(25)["word"])
b = set(keys_alt[keys_alt["over_used_in"] == "A"].head(25)["word"])
print(f"survive both references ({len(a & b)}): {', '.join(sorted(a & b))}")
print()
print(f"only against the chosen reference ({len(a - b)}): {', '.join(sorted(a - b))}")
print()
print("Build your argument on the first list. Mention the second as sensitive")
print("to the reference corpus — which is a limit, not a failure.")

## 9. Write it

Two paragraphs. No more.

**Finding.** What you found, with rates, an effect size, and the raw counts.
Name both corpora. One sentence on what the concordance showed.

**Limits.** The four sentences from 11b: what the corpus is, who is absent, what
the method cannot see, the alternative you cannot rule out.

In [ ]:
print(f"""
FINDING

In {len(target):,} {TARGET_VALUE} documents compared with {len(reference):,}
{ref_label} documents from the same corpus, '{CHECK}' occurs at
{keys[keys['word'] == CHECK]['per_1000_a'].iloc[0]:.2f} per 1,000 words against
{keys[keys['word'] == CHECK]['per_1000_b'].iloc[0]:.2f} (log-likelihood
G² = {keys[keys['word'] == CHECK]['keyness'].iloc[0]:.1f}; raw counts
{int(keys[keys['word'] == CHECK]['count_a'].iloc[0])} and
{int(keys[keys['word'] == CHECK]['count_b'].iloc[0])}). Concordance lines show
[WHAT THE LINES SHOWED]. The result [survives / does not survive] a change of
reference corpus.

LIMITS

[Fill in from 11b: corpus and filter, systematic absence, what counting cannot
see, strongest alternative explanation.]
""")

### 🔧 Your turn 1

Change `TARGET_VALUE` and `REFERENCE_VALUE` to your own comparison and run the
notebook top to bottom.

Everything recomputes. If a cell fails, the usual cause is too few documents on
one side — check the warning in section 2.

### 🔧 Your turn 2

Now switch `CORPUS` to `"colonial"` or `"legal"` and adjust `GROUP_COLUMN` to a
column those tables actually have (`author`, `title`, `licence`).

Running the same workflow on a differently shaped corpus is the test of whether
you have learned the method rather than the notebook.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**On finishing this.** If you have a filled-in finding paragraph and a filled-in
limits paragraph, you have done the work this course exists to teach. The code was
never the hard part — you could have got most of it from a search engine. The hard
parts were: choosing a comparison that a corpus can answer, knowing that the
English stopword list would have done nothing, knowing that `[a-z]+` would have
destroyed the diacritics, and knowing which of your three sentences you could
actually defend.

**Where to take it.** Three routes, in increasing order of effort:

1. **Replace the corpus, keep the workflow.** Your own texts, in a CSV with one
   row per document and a column that splits them into two groups. Everything
   here runs unchanged.
2. **Replicate across languages.** The four Nigerian languages in this corpus
   support the same comparison four times. A finding that holds in all four is a
   different order of claim from one that holds in English.
3. **Publish the negative result.** The measurements in 03a and 03b — that
   standard tooling does nothing useful for Yoruba, Igbo and Hausa — are a
   contribution in themselves, and they are under-documented. You now have the
   code that produces them.

**One last habit.** Whatever you publish, publish the corpus-building script and
the parameters: the MFW count, the `min_count`, the seed, the window. That is what
makes it reproducible, and it is what most of the literature you will read does
not do.

</details>

## Where this stops

It does not. This is the point where the method becomes yours.